In [6]:
from pathlib import Path
import pandas as pd

In [7]:
# ============================================================
# File paths
# ============================================================

SUMMARY_CSV_PATH = Path(
    "/Volumes/LaCie/DAIC-WOZ/Processed Dataset/"
    "manual_interviewer_removal_summary.csv"
)

TRAIN_CSV_PATH = Path(
    "/Volumes/LaCie/DAIC-WOZ/"
    "train_split_Depression_AVEC2017.csv"
)

DEV_CSV_PATH = Path(
    "/Volumes/LaCie/DAIC-WOZ/"
    "dev_split_Depression_AVEC2017.csv"
)

TEST_CSV_PATH = Path(
    "/Volumes/LaCie/DAIC-WOZ/"
    "full_test_split.csv"
)

OUTPUT_CSV_PATH = Path(
    "/Volumes/LaCie/DAIC-WOZ/"
    "DAICWOZ_Initial.csv"
)


# Participants to exclude from the final dataset
EXCLUDED_PARTICIPANT_IDS = {
    321,  # train
    457,  # train
    458,  # dev
    463,  # train
    492,  # dev
}


# Requested final column order
OUTPUT_COLUMNS = [
    "Participant_ID",
    "Folder",
    "Audio_Filename",
    "Output_Audio_Path",
    "Transcript_Path",
    "Dataset",
    "Gender",
    "PHQ8_Binary",
    "PHQ8_Score",
]

In [8]:
# ============================================================
# Helper functions
# ============================================================

def validate_file_exists(file_path: Path) -> None:
    """Raise an error when a required CSV file does not exist."""
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required CSV file was not found:\n{file_path}"
        )

def build_transcript_path(
    output_audio_path: str | Path,
) -> str:
    """
    Build the transcript path corresponding to a processed
    DAIC-WOZ audio recording.

    Example
    -------
    Input:
        .../303_AUDIO_removed_interviewer_manually_cleaned.wav

    Output:
        .../303_TRANSCRIPT.csv
    """

    audio_path = Path(output_audio_path)

    participant_id = audio_path.name.split("_")[0]

    transcript_path = (
        audio_path.parent
        / f"{participant_id}_TRANSCRIPT.csv"
    )

    return str(transcript_path)

def find_column(
    df: pd.DataFrame,
    possible_names: list[str],
    file_label: str,
) -> str:
    """
    Find a column while allowing common capitalization or naming differences.

    For example, this allows:
        Participant_ID
        participant_id
        Participant_ID_Test
    """
    normalized_lookup = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    for possible_name in possible_names:
        normalized_name = possible_name.strip().lower()

        if normalized_name in normalized_lookup:
            return normalized_lookup[normalized_name]

    raise KeyError(
        f"Could not find any of these columns in {file_label}: "
        f"{possible_names}\n"
        f"Available columns: {df.columns.tolist()}"
    )


def normalize_participant_id(series: pd.Series) -> pd.Series:
    """
    Convert participant IDs to pandas nullable integers.

    This handles values such as:
        300
        300.0
        '300'
        '300_P'
    """
    cleaned = (
        series
        .astype("string")
        .str.strip()
        .str.extract(r"(\d+)", expand=False)
    )

    return pd.to_numeric(cleaned, errors="coerce").astype("Int64")


def load_split_metadata(
    csv_path: Path,
    dataset_name: str,
) -> pd.DataFrame:
    """
    Load one DAIC-WOZ split and return standardized metadata columns.
    """
    validate_file_exists(csv_path)

    df = pd.read_csv(csv_path)

    participant_column = find_column(
        df=df,
        possible_names=[
            "Participant_ID",
            "participant_id",
            "Participant_ID_Test",
        ],
        file_label=f"{dataset_name} CSV",
    )

    gender_column = find_column(
        df=df,
        possible_names=["Gender", "gender"],
        file_label=f"{dataset_name} CSV",
    )

    phq8_binary_column = find_column(
        df=df,
        possible_names=[
            "PHQ8_Binary",
            "PHQ8_binary",
            "PHQ_Binary",
        ],
        file_label=f"{dataset_name} CSV",
    )

    phq8_score_column = find_column(
        df=df,
        possible_names=[
            "PHQ8_Score",
            "PHQ8_score",
            "PHQ_Score",
        ],
        file_label=f"{dataset_name} CSV",
    )

    metadata_df = df[
        [
            participant_column,
            gender_column,
            phq8_binary_column,
            phq8_score_column,
        ]
    ].copy()

    metadata_df.columns = [
        "Participant_ID",
        "Gender",
        "PHQ8_Binary",
        "PHQ8_Score",
    ]

    metadata_df["Participant_ID"] = normalize_participant_id(
        metadata_df["Participant_ID"]
    )

    metadata_df["Dataset"] = dataset_name

    # Remove rows without a usable participant ID.
    missing_id_count = int(
        metadata_df["Participant_ID"].isna().sum()
    )

    if missing_id_count > 0:
        print(
            f"Warning: removing {missing_id_count} row(s) from "
            f"{dataset_name} because Participant_ID is missing or invalid."
        )

        metadata_df = metadata_df.loc[
            metadata_df["Participant_ID"].notna()
        ].copy()

    # Remove the requested excluded participants.
    metadata_df = metadata_df.loc[
        ~metadata_df["Participant_ID"].isin(
            EXCLUDED_PARTICIPANT_IDS
        )
    ].copy()

    # Check whether the same participant appears more than once
    # within a split.
    duplicate_ids = (
        metadata_df.loc[
            metadata_df["Participant_ID"].duplicated(keep=False),
            "Participant_ID",
        ]
        .dropna()
        .unique()
        .tolist()
    )

    if duplicate_ids:
        raise ValueError(
            f"Duplicate participant IDs were found in the "
            f"{dataset_name} split:\n{duplicate_ids}"
        )

    return metadata_df[
        [
            "Participant_ID",
            "Dataset",
            "Gender",
            "PHQ8_Binary",
            "PHQ8_Score",
        ]
    ]

In [9]:
# ============================================================
# Main processing
# ============================================================

def create_daicwoz_initial_csv() -> pd.DataFrame:
    """Create the combined DAICWOZ_Initial.csv file."""

    # --------------------------------------------------------
    # 1. Validate and load manual removal summary
    # --------------------------------------------------------

    validate_file_exists(SUMMARY_CSV_PATH)

    summary_df = pd.read_csv(SUMMARY_CSV_PATH)

    status_column = find_column(
        df=summary_df,
        possible_names=["status"],
        file_label="manual interviewer removal summary CSV",
    )

    participant_column = find_column(
        df=summary_df,
        possible_names=["participant_id", "Participant_ID"],
        file_label="manual interviewer removal summary CSV",
    )

    output_audio_path_column = find_column(
        df=summary_df,
        possible_names=[
            "output_audio_path",
            "Output_Audio_Path",
        ],
        file_label="manual interviewer removal summary CSV",
    )

    # Normalize status before comparison so values such as
    # "Success", " success ", and "SUCCESS" are accepted.
    success_mask = (
        summary_df[status_column]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("success")
    )

    successful_audio_df = summary_df.loc[
        success_mask,
        [
            participant_column,
            output_audio_path_column,
        ],
    ].copy()

    successful_audio_df.columns = [
        "Participant_ID",
        "Output_Audio_Path",
    ]

    successful_audio_df["Participant_ID"] = normalize_participant_id(
        successful_audio_df["Participant_ID"]
    )

    successful_audio_df["Output_Audio_Path"] = (
        successful_audio_df["Output_Audio_Path"]
        .astype("string")
        .str.strip()
    )

    # Remove rows missing either participant ID or output path.
    valid_summary_rows = (
        successful_audio_df["Participant_ID"].notna()
        & successful_audio_df["Output_Audio_Path"].notna()
        & successful_audio_df["Output_Audio_Path"].ne("")
    )

    invalid_summary_count = int((~valid_summary_rows).sum())

    if invalid_summary_count > 0:
        print(
            f"Warning: removing {invalid_summary_count} successful "
            f"row(s) because the participant ID or output path is missing."
        )

    successful_audio_df = successful_audio_df.loc[
        valid_summary_rows
    ].copy()

    # Remove excluded participants from the audio summary as well.
    successful_audio_df = successful_audio_df.loc[
        ~successful_audio_df["Participant_ID"].isin(
            EXCLUDED_PARTICIPANT_IDS
        )
    ].copy()

    # --------------------------------------------------------
    # 2. Create Folder, Audio_Filename, and Transcript Path
    # --------------------------------------------------------

    successful_audio_df["Folder"] = (
        successful_audio_df["Output_Audio_Path"]
        .apply(lambda path: str(Path(path).parent))
    )

    successful_audio_df["Audio_Filename"] = (
        successful_audio_df["Output_Audio_Path"]
        .apply(lambda path: Path(path).name)
    )

    successful_audio_df["Transcript_Path"] = (
        successful_audio_df["Output_Audio_Path"]
        .apply(build_transcript_path)
    )

    # Confirm that output files use the .wav extension.
    non_wav_rows = successful_audio_df.loc[
        ~successful_audio_df["Audio_Filename"]
        .str.lower()
        .str.endswith(".wav")
    ]

    if not non_wav_rows.empty:
        print(
            "\nWarning: the following successful rows do not have "
            "a .wav output filename:"
        )
        print(
            non_wav_rows[
                ["Participant_ID", "Output_Audio_Path"]
            ].to_string(index=False)
        )

    # Check whether transcript files exist.
    missing_transcript_rows = successful_audio_df.loc[
        ~successful_audio_df["Transcript_Path"]
        .apply(lambda path: Path(path).exists())
    ]
    
    if not missing_transcript_rows.empty:
        print(
            "\nWarning: the following rows have missing transcript files:"
        )
    
        print(
            missing_transcript_rows[
                [
                    "Participant_ID",
                    "Transcript_Path",
                ]
            ].to_string(index=False)
        )

    # Check for participants with multiple successful audio rows.
    duplicate_audio_participants = (
        successful_audio_df.loc[
            successful_audio_df["Participant_ID"].duplicated(
                keep=False
            ),
            "Participant_ID",
        ]
        .dropna()
        .unique()
        .tolist()
    )

    if duplicate_audio_participants:
        print(
            "\nNote: these participants have multiple successful "
            "audio rows:"
        )
        print(duplicate_audio_participants)

    # --------------------------------------------------------
    # 3. Load train, dev, and test metadata
    # --------------------------------------------------------

    train_metadata_df = load_split_metadata(
        csv_path=TRAIN_CSV_PATH,
        dataset_name="train",
    )

    dev_metadata_df = load_split_metadata(
        csv_path=DEV_CSV_PATH,
        dataset_name="dev",
    )

    test_metadata_df = load_split_metadata(
        csv_path=TEST_CSV_PATH,
        dataset_name="test",
    )

    metadata_df = pd.concat(
        [
            train_metadata_df,
            dev_metadata_df,
            test_metadata_df,
        ],
        ignore_index=True,
    )

    # Ensure a participant does not appear in multiple splits.
    participants_in_multiple_splits = (
        metadata_df.groupby("Participant_ID")["Dataset"]
        .nunique()
    )

    participants_in_multiple_splits = (
        participants_in_multiple_splits.loc[
            participants_in_multiple_splits > 1
        ]
        .index
        .tolist()
    )

    if participants_in_multiple_splits:
        raise ValueError(
            "The following participants appear in more than one "
            f"dataset split:\n{participants_in_multiple_splits}"
        )

    # --------------------------------------------------------
    # 4. Merge audio paths with participant metadata
    # --------------------------------------------------------

    output_df = successful_audio_df.merge(
        metadata_df,
        on="Participant_ID",
        how="left",
        validate="many_to_one",
    )

    # Report successful audio participants that could not be found
    # in any train/dev/test metadata file.
    missing_metadata_df = output_df.loc[
        output_df["Dataset"].isna(),
        [
            "Participant_ID",
            "Output_Audio_Path",
        ],
    ].copy()

    if not missing_metadata_df.empty:
        print(
            "\nWarning: the following successful audio rows did not "
            "match a participant in train, dev, or test:"
        )
        print(missing_metadata_df.to_string(index=False))

    # Keep only participants that matched one of the three splits.
    output_df = output_df.loc[
        output_df["Dataset"].notna()
    ].copy()

    # --------------------------------------------------------
    # 5. Reorder, sort, and save
    # --------------------------------------------------------

    output_df = output_df[OUTPUT_COLUMNS].copy()

    output_df = output_df.sort_values(
        by=["Dataset", "Participant_ID", "Audio_Filename"],
        key=lambda column: (
            column.map({"train": 0, "dev": 1, "test": 2})
            if column.name == "Dataset"
            else column
        ),
        kind="stable",
    ).reset_index(drop=True)

    OUTPUT_CSV_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_df.to_csv(
        OUTPUT_CSV_PATH,
        index=False,
    )

    # --------------------------------------------------------
    # 6. Print summary
    # --------------------------------------------------------

    print("\nDAIC-WOZ initial CSV created successfully.")
    print(f"Output path: {OUTPUT_CSV_PATH}")
    print(f"Total rows: {len(output_df)}")

    print("\nRows by dataset:")
    print(
        output_df["Dataset"]
        .value_counts()
        .reindex(["train", "dev", "test"], fill_value=0)
        .to_string()
    )

    print("\nUnique participants by dataset:")
    print(
        output_df.groupby("Dataset")["Participant_ID"]
        .nunique()
        .reindex(["train", "dev", "test"], fill_value=0)
        .to_string()
    )

    print("\nExcluded participant IDs:")
    print(sorted(EXCLUDED_PARTICIPANT_IDS))

    return output_df

In [10]:
# ============================================================
# Run
# ============================================================

if __name__ == "__main__":
    daicwoz_initial_df = create_daicwoz_initial_csv()

    print("\nFirst five rows:")
    print(daicwoz_initial_df.head().to_string(index=False))


DAIC-WOZ initial CSV created successfully.
Output path: /Volumes/LaCie/DAIC-WOZ/DAICWOZ_Initial.csv
Total rows: 181

Rows by dataset:
Dataset
train    103
dev       32
test      46

Unique participants by dataset:
Dataset
train    103
dev       32
test      46

Excluded participant IDs:
[321, 457, 458, 463, 492]

First five rows:
 Participant_ID                                          Folder                                     Audio_Filename                                                                                  Output_Audio_Path                                                    Transcript_Path Dataset  Gender  PHQ8_Binary  PHQ8_Score
            303 /Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_P 303_AUDIO_removed_interviewer_manually_cleaned.wav /Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_P/303_AUDIO_removed_interviewer_manually_cleaned.wav /Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_P/303_TRANSCRIPT.csv   train       0            0           0
            304 /Volume